# Mutual Fund ETL Pipeline & Data Dictionary Generation
This notebook demonstrates loading cleaned mutual fund CSV datasets into a SQLite database, modeling them into star-schema dimension and fact tables, verifying the integrity of row counts, and dynamically generating a data dictionary markdown document.

## Section 1 — Load Clean CSVs into SQLite

In [1]:
import pandas as pd
from sqlalchemy import create_engine
import json

# SQLite DB
engine = create_engine("sqlite:///../data/db/bluestock_mf.db")
# Load CSVs
fund_master = pd.read_csv("../data/processed/clean_01_fund_master.csv")
nav = pd.read_csv("../data/processed/clean_02_nav_history.csv")
aum = pd.read_csv("../data/processed/clean_03_aum_by_fund_house.csv")
sip = pd.read_csv("../data/processed/clean_04_monthly_sip_inflows.csv")
category = pd.read_csv("../data/processed/clean_05_category_inflows.csv")
folios = pd.read_csv("../data/processed/clean_06_industry_folio_count.csv")
performance = pd.read_csv("../data/processed/clean_07_scheme_performance.csv")
transactions = pd.read_csv("../data/processed/clean_08_investor_transactions.csv")
holdings = pd.read_csv("../data/processed/clean_09_portfolio_holdings.csv")
benchmark = pd.read_csv("../data/processed/clean_10_benchmark_indices.csv")

## Section 2 — Create Tables using Schema.sql

In [1]:
from sqlalchemy import create_engine
import sqlite3

engine = create_engine(
    "sqlite:///../data/db/bluestock_mf.db"
)

with sqlite3.connect(
    "../data/db/bluestock_mf.db"
) as conn:

    with open("../sql/schema.sql") as f:
        conn.executescript(f.read())

OperationalError: no such column: amfi_code

## Section 3 — Automated Append Loads via JSON Configuration

In [3]:
# Define structural JSON string matching configurations
config_json = """
{
    "database_engine": {
        "type": "sqlite",
        "connection_string_template": "sqlite:///../data/db/bluestock_mf.db"
    },
    "tables_configuration": [
        {"table_name": "dim_fund", "dataframe_source": "dim_fund", "sql_parameters": {"if_exists": "append", "index": false}},
        {"table_name": "dim_date", "dataframe_source": "dim_date", "sql_parameters": {"if_exists": "append", "index": false}},
        {"table_name": "fact_nav", "dataframe_source": "fact_nav", "sql_parameters": {"if_exists": "append", "index": false}},
        {"table_name": "fact_transactions", "dataframe_source": "fact_transactions", "sql_parameters": {"if_exists": "append", "index": false}},
        {"table_name": "fact_performance", "dataframe_source": "performance", "sql_parameters": {"if_exists": "append", "index": false}},
        {"table_name": "fact_aum", "dataframe_source": "fact_aum", "sql_parameters": {"if_exists": "append", "index": false}}
    ]
}
"""

# Parse JSON data structure
config = json.loads(config_json)

# Reference active workspace dataframes safely
active_dataframes = {
    "dim_fund": dim_fund,
    "dim_date": dim_date,
    "fact_nav": fact_nav,
    "fact_transactions": fact_transactions,
    "performance": performance,
    "fact_aum": fact_aum
}

# Programmatic data insertion engine execution loop
for target in config["tables_configuration"]:
    source_df = active_dataframes[target["dataframe_source"]]
    sql_args = target["sql_parameters"]
    
    source_df.to_sql(
        target["table_name"],
        engine,
        if_exists=sql_args["if_exists"],
        index=sql_args["index"]
    )
    print(f"Completed processing append pipeline targets for table: {target['table_name']}")

NameError: name 'dim_fund' is not defined

## Verify Row Counts

In [11]:
tables = [
    "dim_fund",
    "dim_date",
    "fact_nav",
    "fact_transactions",
    "fact_performance",
    "fact_aum"
]

for table in tables:
    count = pd.read_sql(
        f"SELECT COUNT(*) AS cnt FROM {table}",
        engine
    )
    print(table)
    print(count)
    print("-"*40)

dim_fund
   cnt
0   80
----------------------------------------
dim_date
    cnt
0  2594
----------------------------------------
fact_nav
     cnt
0  92000
----------------------------------------
fact_transactions
     cnt
0  65556
----------------------------------------
fact_performance
   cnt
0   80
----------------------------------------
fact_aum
   cnt
0  180
----------------------------------------


## Auto Generate data_dictionary.md
Create this after loading the database.

In [12]:
from pathlib import Path

dictionary_path = Path("../data/db/data_dictionary.md")

with open(dictionary_path, "w", encoding="utf-8") as f:

    f.write("# Mutual Fund Data Dictionary\n\n")

    csvs = {
        "clean_01_fund_master": fund_master,
        "clean_02_nav_history": nav,
        "clean_03_aum_by_fund_house": aum,
        "clean_04_monthly_sip_inflows": sip,
        "clean_05_category_inflows": category,
        "clean_06_industry_folio_count": folios,
        "clean_07_scheme_performance": performance,
        "clean_08_investor_transactions": transactions,
        "clean_09_portfolio_holdings": holdings,
        "clean_10_benchmark_indices": benchmark
    }

    for name, df in csvs.items():

        f.write(f"## {name}\n\n")

        f.write("| Column | Data Type |\n")
        f.write("|---------|-----------|\n")

        for col in df.columns:
            f.write(
                f"| {col} | {df[col].dtype} |\n"
            )

        f.write("\n\n")

In [13]:
print("DIM_FUND DUPLICATES")
print(dim_fund["amfi_code"].duplicated().sum())

print()

print("DIM_DATE DUPLICATES")
print(dim_date["date_key"].duplicated().sum())

print()

print("FACT_NAV NULL DATE KEYS")
print(fact_nav["date_key"].isna().sum())

print()

print("FACT_TX NULL DATE KEYS")
print(fact_transactions["date_key"].isna().sum())

print()

print("FACT_AUM NULL DATE KEYS")
print(fact_aum["date_key"].isna().sum())

print()

print("AMFI TYPES")
print(dim_fund["amfi_code"].dtype)
print(fact_nav["amfi_code"].dtype)
print(fact_transactions["amfi_code"].dtype)
print(performance["amfi_code"].dtype)

DIM_FUND DUPLICATES
0

DIM_DATE DUPLICATES
0

FACT_NAV NULL DATE KEYS
0

FACT_TX NULL DATE KEYS
0

FACT_AUM NULL DATE KEYS
0

AMFI TYPES
int64
int64
int64
int64


In [14]:
import sqlite3

conn = sqlite3.connect("../data/db/bluestock_mf.db")

tables = [
    "dim_fund",
    "dim_date",
    "fact_nav",
    "fact_transactions",
    "fact_performance",
    "fact_aum"
]

for table in tables:
    print(f"\n{table}")
    print("-"*50)

    cur = conn.execute(f"PRAGMA table_info({table})")

    for row in cur.fetchall():
        print(row)

conn.close()


dim_fund
--------------------------------------------------
(0, 'amfi_code', 'BIGINT', 0, None, 0)
(1, 'fund_house', 'TEXT', 0, None, 0)
(2, 'scheme_name', 'TEXT', 0, None, 0)
(3, 'category', 'TEXT', 0, None, 0)
(4, 'sub_category', 'TEXT', 0, None, 0)
(5, 'plan', 'TEXT', 0, None, 0)
(6, 'launch_date', 'TEXT', 0, None, 0)
(7, 'benchmark', 'TEXT', 0, None, 0)
(8, 'expense_ratio_pct', 'FLOAT', 0, None, 0)
(9, 'exit_load_pct', 'FLOAT', 0, None, 0)
(10, 'min_sip_amount', 'BIGINT', 0, None, 0)
(11, 'min_lumpsum_amount', 'BIGINT', 0, None, 0)
(12, 'fund_manager', 'TEXT', 0, None, 0)
(13, 'risk_category', 'TEXT', 0, None, 0)
(14, 'sebi_category_code', 'TEXT', 0, None, 0)

dim_date
--------------------------------------------------
(0, 'full_date', 'DATETIME', 0, None, 0)
(1, 'date_key', 'BIGINT', 0, None, 0)
(2, 'year', 'INTEGER', 0, None, 0)
(3, 'quarter', 'INTEGER', 0, None, 0)
(4, 'month', 'INTEGER', 0, None, 0)
(5, 'month_name', 'TEXT', 0, None, 0)
(6, 'day', 'INTEGER', 0, None, 0)

fact_